In [ ]:
import pandas as pd

template_id: <br>
used template | data leo | data aya <br>
3             | 1        | 2 <br>
9             | 2        | 9 <br>
10            | 3        | 10

In [ ]:
#load datasets accuracy per token from the two models
df_leo = pd.read_json('accuracy_per_token_leomistral.json')
df_aya = pd.read_json('accuracy_per_token_ayamodel.json')

#load data with poems
df_results_leo = pd.read_json('500_samples_RESULTS_extracted.json',lines=True)
df_results_aya = pd.read_json('500samples_RESULTS_aya23_extracted.json',lines=True)

In [ ]:
# Extract rows with the highest accuracy for each template
highest_accuracy_rows_leo = df_leo.loc[df_leo.groupby('template')['accuracy'].idxmax()]
highest_accuracy_rows_aya = df_aya.loc[df_aya.groupby('template')['accuracy'].idxmax()]
print(highest_accuracy_rows_aya)
print(highest_accuracy_rows_leo)

In [ ]:
# Get all unique years mentioned in the 500 samples data
# Specify the year of interest
year_of_interest = 1782

# Get all unique years and sort them
unique_years = sorted(df_results_leo['year'].unique())

# Find the index of the given year in the sorted list
try:
    idx = unique_years.index(year_of_interest)
except ValueError:
    print(f"Year {year_of_interest} is not in the data.")
else:
    # Identify the years: given year, year before, and year after
    years_to_filter = [unique_years[idx]]
    if idx > 0:
        years_to_filter.append(unique_years[idx - 1])  # Year before
    if idx < len(unique_years) - 1:
        years_to_filter.append(unique_years[idx + 1])  # Year after

    # Filter the DataFrame
    filtered_df_leo = df_results_leo[df_results_leo['year'].isin(years_to_filter)]
    filtered_df_aya = df_results_aya[df_results_aya['year'].isin(years_to_filter)]


In [ ]:
# For leo-mistral filter out rows where templated_id is either 1 or 3
filtered_df_leo = filtered_df_leo[~filtered_df_leo['template_id'].isin([1, 3])]
filtered_df_aya

In [ ]:
rows_per_year = filtered_df_leo.groupby('year')['labels'].count()
print(rows_per_year)

In [ ]:
# Count tokens per year

# tokens per year in leo model, 
tokens_per_year = filtered_df_leo.groupby('year').apply(lambda x: x['labels'].str.len().sum())
print('tokens per year:')
print(tokens_per_year)


In [ ]:
# Function to get unique labels and associated POS tags for each year
def get_labels_pos_tags(df):
    result = {}
    for year, group in df.groupby('year'):
        # Initialize a dictionary to hold labels and their associated POS tags
        labels_pos_tags = {}
        
        # Iterate over each row in the grouped data
        for _, row in group.iterrows():
            for label, pos in zip(row['labels'], row['extracted_pos_tags']):
                if label not in labels_pos_tags:
                    labels_pos_tags[label] = set()
                labels_pos_tags[label].add(pos)
        
        # Convert sets to lists for readability
        labels_pos_tags = {label: list(pos_tags) for label, pos_tags in labels_pos_tags.items()}
        result[year] = labels_pos_tags
    
    return result

# Get unique labels and associated POS tags for each year for leo model
labels_pos_by_year = get_labels_pos_tags(filtered_df_leo)

# Display the result for leo model
for year, labels in labels_pos_by_year.items():
    print(f"Year: {year}")
    for label, pos_tags in labels.items():
        print(f"  Label: {label} -> POS tags: {pos_tags}")

In [ ]:
# Get unique labels and associated POS tags for each year for aya model
def get_labels_pos_tags_by_template(df):
    result = {}
    for (year, template_id), group in df.groupby(['year', 'template_id']):
        # Initialize a dictionary to hold labels and their associated POS tags
        labels_pos_tags = {}
        
        # Iterate over each row in the grouped data
        for _, row in group.iterrows():
            for label, pos in zip(row['labels'], row['extracted_pos_tags']):
                if label not in labels_pos_tags:
                    labels_pos_tags[label] = set()
                labels_pos_tags[label].add(pos)
        
        # Convert sets to lists for readability
        labels_pos_tags = {label: list(pos_tags) for label, pos_tags in labels_pos_tags.items()}
        
        # Organize the results by year and template ID
        if year not in result:
            result[year] = {}
        result[year][template_id] = labels_pos_tags
    
    return result

# Get unique labels and associated POS tags for each year and template for aya model
labels_pos_by_year_template = get_labels_pos_tags_by_template(filtered_df_aya)

# Display the result
for year, templates in labels_pos_by_year_template.items():
    print(f"Year: {year}")
    for template_id, labels in templates.items():
        print(f"  Template ID: {template_id}")
        for label, pos_tags in labels.items():
            print(f"    Label: {label} -> POS tags: {pos_tags}")